In [1]:
import pandas as pd
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats.mstats import winsorize
from scipy.stats import invwishart
from numpy.linalg import inv
import matplotlib.patches as mpatches

plt.style.use('seaborn-v0_8-whitegrid')
def pivot(variable):
    return raw_df.pivot(index='year', columns='country', values=variable)
def add_const(df):
    return sm.add_constant(df)
def melt(df):
    return df.melt(ignore_index=False).reset_index().set_index(['country','year'])
def winsor(df):
    return df.apply(lambda x: winsorize(x, limits=(.05,.05)))

In [2]:
def significance_stars(pval):
    if pval < 0.01:
        return '***'
    elif pval < 0.05:
        return '**'
    elif pval < 0.1:
        return '*'
    else:
        return ''

def display(fit):
    df = pd.DataFrame({'coefficient': fit.params, 'pvalue': fit.pvalues})
    df['significance'] = df['pvalue'].apply(significance_stars)
    df['results'] = df.apply(
        lambda x: f"{x['coefficient']:.3f}{x['significance']}", axis=1)
    return df[['results']]


def display_w_std_errors(fit):
    df = pd.DataFrame({'coefficient': fit.params,
                       'pvalue': fit.pvalues,
                       'std_error':round(fit.std_errors,3)})
    df['significance'] = df['pvalue'].apply(significance_stars)
    df['results'] = df.apply(
        lambda x: f"{x['coefficient']:.3f}{x['significance']}", axis=1)
    df['results_w_errors'] = df['results'] + ' (' + df['std_error'].astype(str) + ')'
    return df[['results_w_errors']]

    
def base_country_data(target_df_input):
    target_df = target_df_input.copy()
    base_target_df = target_df.copy()
    base_target_df[:] = np.nan
    target_df['HYBRID'] = (target_df['FRA']+target_df['USA'])/2
    for country in base_df.columns:
        for year in range(1870,2020+1):
            if is_base_df.loc[year, country] == 1:
                base_country = country
            else:
                base_country = base_df.loc[year, country]
            if base_country in ['GBR','FRA','USA','HYBRID','DEU']:
                base_value = target_df.loc[year, base_country]
                base_target_df.loc[year, country] = base_value
    return base_target_df

In [3]:
## Data Import

In [4]:
raw_df = pd.read_excel('datasets/JSTdatasetR6.xlsx'
                       ).drop(columns=['country','ifs']).rename(columns={'iso':'country'})
raw_oil_shocks_df = pd.read_excel('datasets/BH2_supply_shocks.xlsx',header=1,usecols=[1])
raw_kaopen_df = pd.read_csv('datasets/KAOPEN.csv',usecols=[0,1,2])
# -------------------------------------------------------------------
credibility_df = pd.read_excel('datasets/CBIData_Romelli_2024.xlsx',
                               sheet_name=1,usecols=['year','iso_a3','cbie_index', 'cbie_policy', 'cbie_finindep']).rename(columns={'iso_a3':'country'})
raw_df = pd.merge(raw_df, credibility_df, how='left',on=['year','country'])
credibility_df = df = pivot('cbie_index')
credibility_relative_df = credibility_df.sub(credibility_df.median(axis=1),axis=0)
credibility_of_policy_df = df = pivot('cbie_policy')
credibility_relative_of_policy_df = credibility_of_policy_df.sub(credibility_of_policy_df.median(axis=1),axis=0)
credibility_of_fin_indep_df = df = pivot('cbie_finindep')
credibility_relative_of_fin_indep_df = credibility_of_fin_indep_df.sub(credibility_of_fin_indep_df.median(axis=1),axis=0)
# -------------------------------------------------------------------
bvx_crises = pd.read_csv('datasets/bvx_crisis.csv',index_col=['country','year'])

In [5]:
## Data Cleaning

In [6]:
cpi_df = pivot('cpi')
gdp_df = pivot('rgdpbarro')
stir_df = pivot('stir')
crisis_df = pivot('crisisJST')
xrate_df = pivot('xrusd')
peg_df = pivot('peg')
base_df = pivot('peg_base')
# ------------------------------------------------------------------------------------------
raw_oil_shocks_df.index = pd.date_range(start='1975-02-01', periods=len(raw_oil_shocks_df), freq='ME')
raw_oil_shocks_df = raw_oil_shocks_df.reindex(
    pd.date_range(start='1975-01-01', periods=len(raw_oil_shocks_df)+1,freq='ME'))
oil_shocks = raw_oil_shocks_df.resample('YE').mean()*12
oil_shocks.index = range(1975,1975+len(oil_shocks))
oil_shocks_df  = cpi_df.copy()
oil_shocks_df[:] = np.nan
for country in oil_shocks_df.columns:
    oil_shocks_df[country] = oil_shocks['oil supply shocks']
# -----------------------------------------------------------------------------------------------
kaopen_df = raw_kaopen_df.pivot(index='year', columns='country', values='kopen')
#-------------------------------------------------------------
investment_df = pivot('iy')
ltrate_df  = pivot('ltrate')
house_price_growth_df = winsor(pivot('hpnom').pct_change(fill_method=None)*100)
unemp_df = pivot('unemp')
public_debt_df = pivot('debtgdp')
public_debt_over_90_df = (public_debt_df>0.9)
public_debt_over_100_df = (public_debt_df>1)
total_loans_df = winsor(pivot('tloans').pct_change(fill_method=None)*100)
mortgage_df = winsor(pivot('tmort').pct_change(fill_method=None)*100)
households_loans_df = winsor(pivot('thh').pct_change(fill_method=None)*100)
business_loans_df = winsor(pivot('tbus').pct_change(fill_method=None)*100)
corporate_debt_df = winsor(pivot('bdebt').pct_change(fill_method=None)*100)
equity_return_df = winsor(pivot('eq_tr'))
housing_return_df = winsor(pivot('housing_tr'))
equity_capgain_df = winsor(pivot('eq_capgain'))
housing_capgain_df = winsor(pivot('housing_capgain'))
wealth_return_df = winsor(pivot('capital_tr'))
risky_return_df = winsor(pivot('risky_tr'))
safe_return_df = winsor(pivot('safe_tr'))
capital_ratio_df = pivot('lev')
loans_to_deposits_df = pivot('ltd')
noncore_funding_df = pivot('noncore')
peg_df = pivot('peg')
peg_strict_df = pivot('peg_strict')
#--------------------------------------------------------------------------------
gdp_nom_df = pivot('gdp')
house_price_to_gdp_df = winsor(pivot('hpnom')/gdp_nom_df)
total_loans_to_gdp_df = winsor(pivot('tloans')/gdp_nom_df)
mortgage_to_gdp_df = winsor(pivot('tmort')/gdp_nom_df)
household_loans_to_gdp_df = winsor(pivot('thh')/gdp_nom_df)
business_loans_to_gdp_df = pivot('tbus')/gdp_nom_df
corporate_debt_to_gdp_df = winsor(pivot('bdebt')/gdp_nom_df)
openness_df = pivot('imports')/gdp_nom_df
credit_growth_df = pivot('tloans')/cpi_df
for country in credit_growth_df.columns:
    credit_growth_df[country] = np.log(credit_growth_df[country])
credit_growth_df = credit_growth_df.diff()
#--------------------------------------------------------------------------------
mortgage_to_gdp_df = winsor(pivot('tmort')/pivot('gdp'))


In [7]:
is_base_df = cpi_df.copy()
is_base_df[:] = 0
for year in range(1870,2020+1):
    base_countries = base_df.loc[year].unique()
    for base_country in base_countries:
        if base_country in ['GBR','FRA','USA','HYBRID','DEU']:
            is_base_df.loc[year, base_country] = 1

is_base_df = is_base_df.drop('HYBRID', axis=1)
kaopen_or_base_df = (is_base_df+kaopen_df).clip(upper=1)

In [8]:
panel_df = raw_df.set_index(['country', 'year'])
panel_df['year'] = panel_df.reset_index()['year'].values
# inflation_delta = winsor((cpi_df.pct_change()*100).sub((cpi_df.pct_change()*100).median(axis=1),axis=0))
inflation_delta = winsor((cpi_df.pct_change()*100).diff(4)) 
# inflation_delta = demand_shock_contributions_df
# inflation_delta = supply_shock_contributions_df
# inflation_delta = winsor(cpi_df.pct_change()*100)  
# inflation_delta = structural_supply_shock_df.copy()
# inflation_delta = structural_demand_shock_df.copy()
base_inflation_delta = base_country_data(inflation_delta) * kaopen_or_base_df
covariates = []
crises_across_horizon_df = (crisis_df==1)|(crisis_df.shift(-1)==1)|(crisis_df.shift(-2)==1)
panel_df['crisis'] = melt(crises_across_horizon_df)
# panel_df['crisis'] = melt(crisis_df)
# panel_df['crisis'] = melt(((winsor(pivot('wage'))/winsor(cpi_df)).pct_change()*100))
# panel_df['crisis'] = melt(winsor((pivot('wage')).pct_change()*100))
# panel_df['crisis'] = melt(winsor((pivot('tloans')/gdp_nom_df).pct_change()*100))
# panel_df['crisis'] = melt(winsor((pivot('thh')/gdp_nom_df).pct_change()*100))
# panel_df['crisis'] = melt(winsor((pivot('wage')/cpi_df).pct_change()*100))
# panel_df['crisis'] = melt(winsor(credit_growth_df))
# panel_df['crisis'] = melt(winsor((pivot('bond_tr')/gdp_nom_df).pct_change()*100))
# panel_df['crisis'] = (bvx_crises==1)|(bvx_crises.shift(-1)==1)|(bvx_crises.shift(-2)==1)

infl_lags = [1]

for lag in range(0,max(4+1,infl_lags[-1]+1)):
    panel_df[f'infl_d({lag})'] = melt(winsor((cpi_df.pct_change()*100).diff()).shift(lag))
    covariates.append(f'infl_d({lag})')
    panel_df[f'gdp({lag})'] = melt(winsor(gdp_df.pct_change(fill_method=None)*100).shift(lag))
    covariates.append(f'gdp({lag})')
# base_inflation_delta_supply = base_country_data(structural_supply_shock_df.rolling(4).sum()) * kaopen_or_base_df
# base_inflation_delta_demand = base_country_data(structural_demand_shock_df.rolling(4).sum()) * kaopen_or_base_df
x_variables = []
# for infl_lag in infl_lags:
#     panel_df[f'supply_infl_delta_lag_{infl_lag}_interaction'] = melt((is_base_df*base_inflation_delta_supply).shift(infl_lag))
#     x_variables.append(f'supply_infl_delta_lag_{infl_lag}_interaction')
# for infl_lag in infl_lags:
#     panel_df[f'demand_infl_delta_lag_{infl_lag}_interaction'] = melt((is_base_df*base_inflation_delta_demand).shift(infl_lag))
#     x_variables.append(f'demand_infl_delta_lag_{infl_lag}_interaction')
for infl_lag in infl_lags:
    panel_df[f'delta_lag_{infl_lag}_interaction'] = melt((is_base_df*base_inflation_delta).shift(infl_lag))
    x_variables.append(f'delta_lag_{infl_lag}_interaction')
for infl_lag in infl_lags:
    # panel_df[f'supply_infl_delta_lag_{infl_lag}'] = melt(base_inflation_delta_supply.shift(infl_lag))
    # x_variables.append(f'supply_infl_delta_lag_{infl_lag}')
    # panel_df[f'demand_infl_delta_lag_{infl_lag}'] = melt(base_inflation_delta_demand.shift(infl_lag))
    # x_variables.append(f'demand_infl_delta_lag_{infl_lag}')
    panel_df[f'infl_delta_lag_{infl_lag}'] = melt(base_inflation_delta.shift(infl_lag))
    x_variables.append(f'infl_delta_lag_{infl_lag}')
    panel_df[f'base_lag_{infl_lag}'] = melt(is_base_df.shift(infl_lag))
    x_variables.append(f'base_lag_{infl_lag}')
x_variables = x_variables  + covariates
years_to_verify = infl_lags[-1]
is_and_was_base_or_peg_df = (peg_df.rolling(years_to_verify+1).min().fillna(0)==1)|(is_base_df.rolling(years_to_verify+1).min().fillna(0)==1)
panel_df['is_and_was_base_or_peg'] = melt(is_and_was_base_or_peg_df)
sub_sample = panel_df[(panel_df.year >= 1870)&(panel_df.year < 2020)]
sub_sample = sub_sample[~sub_sample.year.isin(range(1914,1918+1))&~sub_sample.year.isin(range(1939,1945+1))]
# sub_sample = sub_sample[~sub_sample.year.isin(range(1918,1925+1))&~sub_sample.year.isin(range(1945,1950+1))] 
sub_sample = sub_sample[sub_sample.is_and_was_base_or_peg]
for infl_lag in infl_lags:
    sub_sample.loc[sub_sample.peg_type == 'BASE', f'infl_d_{infl_lag}'] = 0
fit = PanelOLS(sub_sample['crisis'], add_const(sub_sample[x_variables]), 
               entity_effects=True,time_effects=False).fit(cov_type="clustered", cluster_entity=True)
display(fit)
# display_w_std_errors(fit)
# fit

c:\Users\james\anaconda3\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


,results
const,0.137***
delta_lag_1_interaction,0.012***
infl_delta_lag_1,-0.009**
base_lag_1,-0.084***
infl_d(0),-0.001
gdp(0),-0.010**
infl_d(1),-0.002
gdp(1),-0.000
infl_d(2),-0.000
gdp(2),0.001
